# 🎙️ Voice Over BYD dengan Kokoro
Jalankan sel **satu per satu dari atas** dengan tombol ▶ di kiri setiap sel.

⚠️ Kokoro tidak resmi mendukung bahasa Indonesia. Dengarkan sampel dulu sebelum memakai hasilnya.

## Langkah 1 — Instal (±1–3 menit, cukup sekali)
Kalau berhasil, di akhir muncul **✅ Instal berhasil** beserta deretan simbol bunyi. Kalau yang muncul error merah, **berhenti** dan kirim teks errornya.

In [ ]:
# Instal tanpa 'spacy' (penyebab error build). Kita tidak butuh modul bahasa Inggris.
!apt-get -qq -y install espeak-ng > /dev/null
!pip install -q --no-deps kokoro==0.9.4 misaki==0.9.4
!pip install -q loguru num2words phonemizer-fork espeakng-loader addict soundfile huggingface_hub

import importlib.util, sys, types
for paket in ['torch', 'transformers']:
    if importlib.util.find_spec(paket) is None:
        !pip install -q {paket}
if importlib.util.find_spec('spacy') is None:
    sys.modules['spacy'] = types.ModuleType('spacy')  # pengganti kosong, tidak dipakai

from kokoro import KPipeline
from misaki import espeak
print('✅ Instal berhasil:', espeak.EspeakG2P(language='id')('Halo, apa kabar?')[0])

## Langkah 2 — Naskah & perbaikan pengucapan
Kalau ada kata yang salah ucap, tambahkan ke `PENGUCAPAN` dengan format `"kata asli": "ejaan bunyi"`, lalu jalankan ulang sel ini.

In [ ]:
import importlib.util, sys, types
if importlib.util.find_spec('spacy') is None:
    sys.modules['spacy'] = types.ModuleType('spacy')
import json, re, zipfile
import numpy as np, soundfile as sf
from IPython.display import Audio, display
from kokoro import KPipeline
from misaki import espeak
SR = 24000

NASKAH = [
 {
  "id": 1,
  "title": "Pembuka",
  "text": "Ketika merek mobil listrik masuk ke pasar baru, pertanyaan pertamanya bukan: mobil apa yang mau dijual? Tetapi: apa yang sebenarnya dibutuhkan pasar? Video ini membahas bagaimana BYD di Indonesia memperoleh dan menggunakan informasi pasar untuk mendukung keberlanjutan bisnisnya."
 },
 {
  "id": 2,
  "title": "Profil perusahaan",
  "text": "BYD didirikan pada 1995 di Shenzhen, Tiongkok, dan berawal dari bisnis baterai sebelum masuk ke kendaraan listrik. Di Indonesia, BYD hadir lebih dulu lewat taksi listrik Bluebird sejak 2019 dan bus listrik Transjakarta sejak 2022. Untuk mobil penumpang, BYD resmi masuk pada Januari 2024 dengan tiga model: Dolphin, Atto 3, dan Seal."
 },
 {
  "id": 3,
  "title": "Masalah pasar",
  "text": "Tetapi masuk ke pasar tidak otomatis membuat konsumen percaya. Survei Tirto bersama Jakpat pada Juli 2026 menunjukkan kekhawatiran terbesar calon pembeli mobil listrik adalah baterai, yaitu 28,65 persen. Berikutnya penggunaan sehari-hari, seperti jarak tempuh dan pengisian daya, sebesar 22,59 persen. Ini bukan riset internal BYD, melainkan data sekunder yang membantu kita membaca lingkungan pemasaran."
 },
 {
  "id": 4,
  "title": "Sistem informasi pemasaran",
  "text": "Lalu, bagaimana perusahaan memperoleh informasi pasar? Menurut Kotler dan rekan, sistem informasi pemasaran terdiri dari catatan internal, intelijen pemasaran, dan riset pemasaran. Pada BYD, ketiganya terlihat dari aktivitas yang terdokumentasi. Formulir test drive di situs resmi mengumpulkan nama, kontak, dan area dealer, yaitu data prospek pelanggan. Showroom dan media sosial menjadi sumber intelijen tentang minat konsumen dan pesaing. Survei eksternal melengkapi gambaran pasar."
 },
 {
  "id": 5,
  "title": "Metode riset",
  "text": "Dari sisi metode, Laporan Keberlanjutan BYD 2024 menyatakan bahwa analisis kelayakan pada tahap riset dan pengembangan mencakup kebutuhan pelanggan, tren pasar, serta riset pasar dan analisis profesional. Namun, rincian desain risetnya tidak dipublikasikan. Jadi, metode spesifiknya kita identifikasi dari aktivitas perusahaan: pameran dan test drive sebagai observasi dan data primer, formulir test drive sebagai data perilaku, dan survei pihak ketiga sebagai data sekunder. Riset yang baik memang menggunakan lebih dari satu metode."
 },
 {
  "id": 6,
  "title": "Lingkungan pemasaran",
  "text": "Informasi itu kemudian dibaca sebagai insight tentang lingkungan pemasaran. Dari sisi ekonomi, daftar harga resmi BYD mencantumkan tanggal, sehingga harga harus dibaca sesuai periodenya. Dari sisi persaingan, VinFast dan Chery membidik pasar yang sama. Dari sisi infrastruktur, ketersediaan SPKLU menentukan kemudahan pemakaian sehari-hari."
 },
 {
  "id": 7,
  "title": "Riset & pengembangan produk",
  "text": "Bagaimana informasi pasar terhubung dengan produk? Contohnya Blade Battery. Saat peluncuran pada Maret 2020, BYD menyatakan baterai ini dikembangkan selama beberapa tahun untuk meredam kekhawatiran soal keamanan baterai. Menurut BYD, baterai ini tidak berasap atau terbakar dalam uji tusuk paku. Alurnya dapat direkonstruksi: kekhawatiran pengguna, informasi pasar, riset dan pengembangan, produk, uji keamanan, lalu komunikasi ke pasar. Ini rekonstruksi dari dokumen publik, bukan bukti adanya survei internal tertentu. Dan survei Tirto 2026 menunjukkan kekhawatiran itu masih relevan."
 },
 {
  "id": 8,
  "title": "Pemasaran berkelanjutan",
  "text": "Bagaimana semua ini mendukung keberlanjutan? Martin dan Schouten memakai kerangka triple bottom line: ekonomi, sosial, dan lingkungan. Kerja sama BYD dengan PLN untuk ekosistem kendaraan listrik dan home charging dapat dibaca sebagai respons atas kekhawatiran soal pengisian daya. Pada 2024, BYD menandatangani pembelian lahan di Subang untuk fasilitas produksi, riset, dan pelatihan. Di Indonesia International Sustainability Forum 2024, BYD menyediakan 18 unit Seal sebagai kendaraan resmi. Semua ini interpretasi atas aktivitas yang terdokumentasi, bukan dampak yang sudah terukur."
 },
 {
  "id": 9,
  "title": "Penutup",
  "text": "Pada akhirnya, riset pemasaran adalah sebuah siklus. Kotler menggambarkannya dalam enam langkah: mendefinisikan masalah, menyusun rencana riset, mengumpulkan informasi, menganalisis, menyajikan temuan, dan mengambil keputusan. Setelah itu, pasar memberi umpan balik dan siklus dimulai lagi. Jadi, pemasaran berkelanjutan bukan hanya menjual mobil listrik, tetapi terus memahami pasar dan belajar dari perubahannya. Terima kasih."
 }
]

PENGUCAPAN = {
 "BYD": "Bi Wai Di",
 "test drive": "tes draif",
 "insight": "insait",
 "Blade Battery": "Bleid Beteri",
 "home charging": "hom carjing",
 "triple bottom line": "tripel botom lain",
 "International Sustainability Forum": "Internesyenel Sastenebiliti Forum",
 "VinFast": "Vinfas",
 "Chery": "Ceri",
 "Schouten": "Skauten",
 "Shenzhen": "Syenzen",
 "Bluebird": "Blubed",
 "Dolphin": "Dolfin",
 "Seal": "Sil",
 "SPKLU": "es pe ka el u",
 "PLN": "pe el en",
 "28,65 persen": "dua puluh delapan koma enam lima persen",
 "22,59 persen": "dua puluh dua koma lima sembilan persen"
}

def perbaiki_ucapan(teks, kamus):
    # Ganti istilah asing/singkatan dengan ejaan bunyi Indonesia (kunci terpanjang dulu).
    for asal in sorted(kamus, key=len, reverse=True):
        teks = re.sub(re.escape(asal), kamus[asal], teks, flags=re.IGNORECASE)
    return teks


def per_kalimat(teks):
    # Jalur espeak di Kokoro bisa memotong teks panjang, jadi pisahkan per kalimat dengan '\n'.
    return "\n".join(s.strip() for s in re.split(r"(?<=[.?!])\s+", teks) if s.strip())


_pipelines = {}


def pipeline_untuk(suara):
    lang = suara[0]
    if lang not in _pipelines:
        p = KPipeline(lang_code=lang, repo_id="hexgrad/Kokoro-82M")
        p.g2p = espeak.EspeakG2P(language="id")  # paksa fonem bahasa Indonesia
        _pipelines[lang] = p
    return _pipelines[lang]


def sintesis(teks, suara, speed=1.0, jeda=0.35):
    p = pipeline_untuk(suara)
    potongan = []
    diam = np.zeros(int(SR * jeda), dtype=np.float32)
    for _, _, audio in p(teks, voice=suara, speed=speed, split_pattern=r"\n+"):
        potongan += [audio.numpy(), diam]
    return np.concatenate(potongan) if potongan else np.zeros(1, dtype=np.float32)


print('✅ Siap:', len(NASKAH), 'scene')

## Langkah 3 — Dengarkan sampel scene 1 dengan 6 suara
Putar setiap pemutar audio di bawah, lalu catat suara mana yang paling enak didengar. (Proses pertama agak lama karena model diunduh.)

In [ ]:
SUARA_SAMPEL = ['ef_dora', 'em_alex', 'if_sara', 'im_nicola', 'pf_dora', 'af_heart']
teks = per_kalimat(perbaiki_ucapan(NASKAH[0]['text'], PENGUCAPAN))
for s in SUARA_SAMPEL:
    audio = sintesis(teks, s)
    print(f'🔊 {s}  ({len(audio)/SR:.1f} detik)')
    display(Audio(audio, rate=SR))

## Langkah 4 — Buat semua scene
Ganti `SUARA` dengan pilihanmu dari Langkah 3. `KECEPATAN`: 1.0 = normal, 0.9 = lebih pelan.

In [ ]:
SUARA = 'ef_dora'
KECEPATAN = 1.0

berkas = []
for sc in NASKAH:
    audio = sintesis(per_kalimat(perbaiki_ucapan(sc['text'], PENGUCAPAN)), SUARA, KECEPATAN)
    nama = f"vo{sc['id']:02d}.wav"
    sf.write(nama, audio, SR); berkas.append(nama)
    print(f"✅ {nama}  {sc['title']:<30} {len(audio)/SR:5.1f} detik")
with zipfile.ZipFile('vo-byd.zip', 'w') as z:
    for b in berkas: z.write(b)
print('Selesai. Jalankan Langkah 5 untuk mengunduh.')

## Langkah 5 — Unduh

In [ ]:
from google.colab import files
files.download('vo-byd.zip')